<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: класс Notification в C#</h2>

----

### Вариант задания 24


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Notification в C#, который будет представлять уведомления
пользователям. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.
Требования к базовому классу Notification:
• Атрибуты: ID уведомления (NotificationId), Текст уведомления (MessageText),
Тип уведомления (Type).
• Методы:
o DisplayNotification(): метод для отображения уведомления
пользователю.
o SendNotification(): метод для отправки уведомления.
o GetNotificationDetails(): метод для получения деталей уведомления.
Требования к производным классам:
1. EmailУведомление (EmailNotification): Должно содержать дополнительные
атрибуты, такие как Адрес электронной почты (EmailAddress).
Метод SendNotification() должен быть переопределен для отправки
уведомления по электронной почте.
2. SMSУведомление (SMSNotification): Должно содержать дополнительные
атрибуты, такие как Номер телефона (PhoneNumber).
Метод SendNotification() должен быть переопределен для отправки
уведомления через SMS.
3. PushУведомление (PushNotification) (если требуется третий класс): Должно
содержать дополнительные атрибуты, такие как Платформа (Platform,
например, iOS или Android). Метод DisplayNotification() должен быть
переопределен для отображения уведомления на мобильной платформе.

Дополнительное задание

Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) используйтие в проекте коллекции, делегаты, события.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void NotificationAction(string message);

public interface IArchivable
{
    bool IsArchived { get; set; }
    void Archive();
}

public class Notification : IArchivable
{
    public int NotificationId { get; set; }
    public string MessageText { get; set; }
    public string Type { get; set; }
    public DateTime Timestamp { get; set; } = DateTime.Now;
    public int Priority { get; set; } = 1;
    public bool IsSent { get; set; }
    public string SenderName { get; set; } = "Система";
    public int RetryLimit { get; set; } = 3;
    public string DeliveryStatus { get; set; } = "Ожидание";
    public string Category { get; set; } = "Общее";

    public event NotificationAction OnSent;
    public event NotificationAction OnArchived;
    public bool IsArchived { get; set; }

    public Notification(int id, string text, string type) => (NotificationId, MessageText, Type) = (id, text, type);

    protected void RaiseOnSent(string message) => OnSent?.Invoke(message);
    protected void RaiseOnArchived(string message) => OnArchived?.Invoke(message);

    public virtual void DisplayNotification() => Console.WriteLine($"[Уведомление #{NotificationId}] От: {SenderName} | Тип: {Type} | Текст: \"{MessageText}\"");
    public virtual void Validate() { DeliveryStatus = "Проверено"; Console.WriteLine($"[Валидация] Проверка #{NotificationId} пройдена."); }
    
    public virtual void SendNotification()
    {
        IsSent = true;
        DeliveryStatus = "Отправлено";
        Console.WriteLine($"[Отправка] Уведомление #{NotificationId} ({Type}) отправлено.");
        RaiseOnSent($"Событие: Уведомление #{NotificationId} отправлено.");
    }

    public virtual void SendNotification(int retries) => Console.WriteLine($"[Повтор #{NotificationId}] Попыток: {retries}/{RetryLimit}");
    public virtual void SendNotification(string header) => Console.WriteLine($"[{header}] Уведомление #{NotificationId}");

    void IArchivable.Archive()
    {
        IsArchived = true;
        DeliveryStatus = "Архивировано";
        Console.WriteLine($"[Архив] Уведомление #{NotificationId} в архиве.");
        RaiseOnArchived($"Событие: Уведомление #{NotificationId} в архиве.");
    }
}

public class EmailNotification : Notification
{
    public string EmailAddress { get; set; }
    public string Subject { get; set; } = "Важное сообщение";
    public bool HasAttachment { get; set; }
    public int SmtpPort { get; set; } = 587;

    public EmailNotification(int id, string text, string email) : base(id, text, "Email") => (EmailAddress, Category) = (email, "Почта");

    public override void SendNotification()
    {
        IsSent = true;
        Console.WriteLine($"[Email] На {EmailAddress} (Тема: \"{Subject}\"): {MessageText}");
        RaiseOnSent($"Событие: Email #{NotificationId} отправлен.");
    }
    public void AttachFile(string file) { HasAttachment = true; Console.WriteLine($"[Email] Прикреплен файл: {file}"); }
}

public class SmsNotification : Notification
{
    public string PhoneNumber { get; set; }
    public string MobileOperator { get; set; } = "Tele2";
    public decimal Cost { get; set; } = 1.50m;

    public SmsNotification(int id, string text, string phone) : base(id, text, "SMS") => (PhoneNumber, Category) = (phone, "SMS");

    public override void SendNotification()
    {
        IsSent = true;
        Console.WriteLine($"[SMS] На номер {PhoneNumber} ({MobileOperator}): {MessageText}");
        RaiseOnSent($"Событие: SMS #{NotificationId} отправлен.");
    }
    public void CheckDelivery() => Console.WriteLine($"[SMS] Запрошен отчет для #{NotificationId}.");
}

public class PushNotification : Notification
{
    public string DeviceToken { get; set; }
    public bool SoundEnabled { get; set; } = true;

    public PushNotification(int id, string text, string token) : base(id, text, "Push") => (DeviceToken, Category) = (token, "Push");

    public override void SendNotification()
    {
        IsSent = true;
        Console.WriteLine($"[Push] Токен {DeviceToken}: {MessageText}");
        RaiseOnSent($"Событие: Push #{NotificationId} отправлен.");
    }
}

public class SlackNotification : Notification
{
    public string ChannelName { get; set; }
    public string BotName { get; set; } = "AlertBot";

    public SlackNotification(int id, string text, string channel) : base(id, text, "Slack") => (ChannelName, Category) = (channel, "Slack");

    public override void SendNotification()
    {
        IsSent = true;
        Console.WriteLine($"[Slack] В канал {ChannelName} от {BotName}: {MessageText}");
        RaiseOnSent($"Событие: Slack #{NotificationId} отправлен.");
    }
}

public class NotificationManager<T> where T : Notification
{
    private List<T> list = new List<T>();
    private Dictionary<int, T> dict = new Dictionary<int, T>();

    public void Add(T item)
    {
        list.Add(item);
        dict[item.NotificationId] = item;
        item.OnSent += msg => Console.WriteLine($"[Лог] {msg}");
    }

    public void ProcessAll()
    {
        foreach (var item in list) { item.DisplayNotification(); item.SendNotification(); Console.WriteLine(); }
    }

    public T FindById(int id) => dict.TryGetValue(id, out var item) ? item : null;
}


var manager = new NotificationManager<Notification>();

var email = new EmailNotification(1, "Привет!", "user@example.com");
email.AttachFile("doc.pdf");

var sms = new SmsNotification(2, "Код: 1234", "+79990000000");
sms.CheckDelivery();

manager.Add(email);
manager.Add(sms);
manager.ProcessAll();

Console.WriteLine($"Поиск по ID (1): {manager.FindById(1)?.Category}\n");

Console.WriteLine("=== Тест DI и архивации ===");
Notification push = new PushNotification(3, "Обновление!", "token_99");
push.Validate();
push.SendNotification();
((IArchivable)push).Archive();

[Email] Прикреплен файл: doc.pdf
[SMS] Запрошен отчет для #2.
[Уведомление #1] От: Система | Тип: Email | Текст: "Привет!"
[Email] На user@example.com (Тема: "Важное сообщение"): Привет!
[Лог] Событие: Email #1 отправлен.

[Уведомление #2] От: Система | Тип: SMS | Текст: "Код: 1234"
[SMS] На номер +79990000000 (Tele2): Код: 1234
[Лог] Событие: SMS #2 отправлен.

Поиск по ID (1): Почта

=== Тест DI и архивации ===
[Валидация] Проверка #3 пройдена.
[Push] Токен token_99: Обновление!
[Архив] Уведомление #3 в архиве.
